In [1]:
import pandas as pd
import spacy
from tqdm import tqdm
from transformers import BertTokenizer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tqdm.pandas()

In [3]:
# Load dataset
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/cleanverses.csv", encoding='utf-8-sig')
print("Initial shape:", df.shape)

Initial shape: (39006, 15)


In [4]:
df = df.dropna(subset=["lyrics"])
print("Shape after dropping NaN:", df.shape)

Shape after dropping NaN: (39005, 15)


In [5]:
# Ensure 'lyrics' column is of string type
df["lyrics"] = df["lyrics"].astype(str)

In [8]:
# Load spaCy model (disable NER and parser for faster processing)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [9]:
# Tokenize lyrics using spaCy
df['tokens'] = [
    [token.text for token in doc] 
    for doc in tqdm(
        nlp.pipe(df['lyrics']), 
        total=len(df), 
        desc="Tokenizing Lyrics"
    )
]

Tokenizing Lyrics:   0%|          | 0/39005 [00:00<?, ?it/s]

Tokenizing Lyrics: 100%|██████████| 39005/39005 [02:55<00:00, 221.86it/s]


In [10]:
# Identify verses with unknown tokens
def has_unknown_tokens(tokens):
    return any(token == '[UNK]' for token in tokens)
df['has_unknown'] = df['tokens'].apply(has_unknown_tokens)

In [11]:
# Identify and remove verses with unknown tokens
unknown_token_count = df['has_unknown'].sum()
print(f"Number of verses with unknown tokens: {unknown_token_count}")

# Filter out verses with unknown tokens
df = df[~df['has_unknown']].reset_index(drop=True)
print("Shape after removing verses with unknown tokens:", df.shape)

Number of verses with unknown tokens: 0
Shape after removing verses with unknown tokens: (39005, 17)


In [12]:
df = df.dropna(subset=["tokens"])
print("Shape after dropping NaN:", df.shape)

Shape after dropping NaN: (39005, 17)


In [13]:
# Save output
print("After tokenizing:", df.shape)
df.to_csv('C:/Users/User/Documents/devanasokan_fyp/preparation/tokenverses.csv', index=False, encoding='utf-8-sig')

After tokenizing: (39005, 17)
